In [1]:
!pip install pandas

In [159]:
import pandas as pd
import sqlite3
from sqlite3 import Error

In [160]:
def create_connection(path):
    connection = None
    try:
        connection = sqlite3.connect(path)
    except Error as e:
        print(f"The error {e} occured")
    return connection

connection = create_connection("../checking-logs.sqlite")

In [161]:
query = """WITH sup_table AS (SELECT uid, labname, CASE WHEN (strftime('%s', first_commit_ts) - strftime('%s', first_view_ts)) > 0 THEN 'after' ELSE 'before' END AS time, 
AVG((strftime('%s', first_commit_ts) - deadlines.deadlines) / 3600) AS hours
FROM test LEFT JOIN deadlines ON labs = labname
WHERE labname <> 'project1'
GROUP BY uid, time),

before_n_after AS (SELECT uid
FROM sup_table
GROUP BY uid
HAVING SUM(time = 'before') > 0 AND SUM(time = 'after') > 0)

SELECT time, AVG(hours) AS avg_diff
FROM sup_table
WHERE uid IN (SELECT uid FROM before_n_after)
GROUP BY time;
"""

pd.read_sql(query, connection)

,time,avg_diff
0,after,-99.523810
1,before,-66.047619


In [162]:
query = """WITH sup_table AS (SELECT uid, labname, CASE WHEN (strftime('%s', first_commit_ts) - strftime('%s', first_view_ts)) > 0 THEN 'after' ELSE 'before' END AS time, 
AVG((strftime('%s', first_commit_ts) - deadlines.deadlines) / 3600) AS hours
FROM control LEFT JOIN deadlines ON labs = labname
WHERE labname <> 'project1'
GROUP BY uid, time),

before_n_after AS (SELECT uid
FROM sup_table
GROUP BY uid
HAVING SUM(time = 'before') > 0 AND SUM(time = 'after') > 0)

SELECT time, AVG(hours) AS avg_diff
FROM sup_table
WHERE uid IN (SELECT uid FROM before_n_after)
GROUP BY time;
"""

pd.read_sql(query, connection)

,time,avg_diff
0,after,-99.322222
1,before,-98.033333


In [163]:
connection.close()